# Train / Validation / Test Split

## Objective

Split the labeled dataset into training, validation, and test sets before performing deep analysis or making modeling decisions.

## Steps

1. Load the labeled ML table artifact from Notebook 2.
2. Check the available date range.
3. Check the label distribution.
4. Decide whether to use a random split or a time-based split.
5. Create the train, validation, and test sets.
6. Validate the size, date range, and label balance of each split.
7. Save the three splits as artifacts.

## Output

- `train.parquet`
- `validation.parquet`
- `test.parquet`

In [1]:
#Load the labeled ML table artifact from Notebook 2.
import pandas as pd
MlTable = pd.read_parquet("../artifacts/ml_table_labeled.parquet")
print("Artifact loaded successfully! Shape:", MlTable.shape)

Artifact loaded successfully! Shape: (99441, 41)


In [5]:
#Check the available date range.
print("Minimum purchase timestamp:", MlTable[['order_purchase_timestamp']].min())
print("Maximum purchase timestamp:", MlTable[['order_purchase_timestamp']].max())


Minimum purchase timestamp: order_purchase_timestamp   2016-09-04 21:15:00
dtype: datetime64[us]
Maximum purchase timestamp: order_purchase_timestamp   2018-10-17 17:30:00
dtype: datetime64[us]


In [6]:
# Check the label distribution.
print("Label distribution:")
print(MlTable["is_late"].value_counts(normalize=True))

Label distribution:
is_late
0    0.92129
1    0.07871
Name: proportion, dtype: float64


In [ ]:
# Decide whether to use a random split or a time-based split.
#Decision: Time-based Split
#Justification:
# chose a time-based split instead of a random split for the following reasons:

#Real-world Simulation: In e-commerce, we always use historical data from the past to train a model that predicts future outcomes.

#Preventing Data Leakage: A random split would mix past and future data together, allowing the model to "see" the future during training, which gives fake high accuracy.

#Chronological Order: Splitting the data based on time (Train on older orders, Test on newer orders) accurately simulates how the model will perform in real-world deployment.

In [13]:
# Create the train, validation, and test sets.

MlTable["order_purchase_timestamp"] = pd.to_datetime(MlTable["order_purchase_timestamp"])
MlTable = MlTable.sort_values(by="order_purchase_timestamp", ascending=True).reset_index(drop=True)
print("Is it sorted now?", MlTable["order_purchase_timestamp"].is_monotonic_increasing)
print("Oldest date (Start):", MlTable["order_purchase_timestamp"].min())
print("Newest date (End):", MlTable["order_purchase_timestamp"].max())

print(MlTable[["order_purchase_timestamp"]].head())
print(MlTable[["order_purchase_timestamp"]].tail())

#  85% Train & Validation

#  15% Test

total_rows = len(MlTable)

train_end = int(total_rows * 0.70)
val_end = int(total_rows * 0.85)  


train_df = MlTable.iloc[:train_end]
val_df = MlTable.iloc[train_end:val_end]
test_df = MlTable.iloc[val_end:]

print(f"Train set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Test set shape: {test_df.shape}")


Is it sorted now? True
Oldest date (Start): 2016-09-04 21:15:00
Newest date (End): 2018-10-17 17:30:00
  order_purchase_timestamp
0      2016-09-04 21:15:00
1      2016-09-05 00:15:00
2      2016-09-13 15:24:00
3      2016-09-15 12:16:00
4      2016-10-02 22:07:00
      order_purchase_timestamp
99436      2018-09-29 09:13:00
99437      2018-10-01 15:30:00
99438      2018-10-03 18:55:00
99439      2018-10-16 20:16:00
99440      2018-10-17 17:30:00
Train set shape: (69608, 41)
Validation set shape: (14916, 41)
Test set shape: (14917, 41)


In [15]:
# Validate the size, date range, and label balance of each split.
print("Train set date range:", train_df["order_purchase_timestamp"].min(), "to", train_df["order_purchase_timestamp"].max())
print("Validation set date range:", val_df["order_purchase_timestamp"].min(), "to", val_df["order_purchase_timestamp"].max())
print("Test set date range:", test_df["order_purchase_timestamp"].min(), "to", test_df["order_purchase_timestamp"].max()) 


print("Train set label distribution:")
print(train_df["is_late"].value_counts(normalize=True)) 
print("Validation set label distribution:")
print(val_df["is_late"].value_counts(normalize=True)) 
print("Test set label distribution:")
print(test_df["is_late"].value_counts(normalize=True))

Train set date range: 2016-09-04 21:15:00 to 2018-04-13 21:50:00
Validation set date range: 2018-04-13 21:53:00 to 2018-06-20 16:28:00
Test set date range: 2018-06-20 16:28:00 to 2018-10-17 17:30:00
Train set label distribution:
is_late
0    0.912596
1    0.087404
Name: proportion, dtype: float64
Validation set label distribution:
is_late
0    0.947305
1    0.052695
Name: proportion, dtype: float64
Test set label distribution:
is_late
0    0.935845
1    0.064155
Name: proportion, dtype: float64


In [16]:
# Save the three splits as artifacts.
from pathlib import Path

Test_path = Path("../artifacts/ml_table_test.parquet")
Train_path = Path("../artifacts/ml_table_train.parquet")
Val_path = Path("../artifacts/ml_table_val.parquet")


# حفظ الجداول كملفات Parquet
train_df.to_parquet(Train_path, index=False)
val_df.to_parquet(Val_path, index=False)
test_df.to_parquet(Test_path, index=False)

print("All data splits (Train, Val, Test) have been saved successfully as parquet artifacts!")

All data splits (Train, Val, Test) have been saved successfully as parquet artifacts!
